# 🤖 BTC Wheel Bot — Complete Strategy Explainer

> **What this notebook covers:** Everything you need to understand how the BTC Wheel Bot works — from first principles of options through to the genetic optimiser results and walk-forward validation. Each section builds on the last. Run every cell top-to-bottom and every chart will populate with **your real live data**.

---
| Section | Topic |
|---------|-------|
| 1 | Setup & data loading |
| 2 | The Wheel Strategy — plain English |
| 3 | Options fundamentals with P&L diagrams |
| 4 | Your active config visualised |
| 5 | Short-put mechanics — strike, premium, breakeven |
| 6 | Delta hedging with perpetual futures |
| 7 | Backtest trade history — 27 real trades |
| 8 | Parameter sweep — sensitivity curves |
| 9 | Genetic algorithm — what the best bots look like |
| 10 | Monte Carlo — 100 simulated futures |
| 11 | Walk-forward validation — does it generalise? |
| 12 | Black swan stress tests |
| 13 | Strategy improvement ideas |

## 1 · Setup & data loading

Run this cell first — it imports all libraries and loads your real data files from `~/Documents/btc-wheel-bot/data/`.

In [ ]:
import json, yaml, warnings, math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.ticker import FuncFormatter
from scipy import stats
from scipy.stats import norm
from pathlib import Path

warnings.filterwarnings('ignore')
plt.style.use('dark_background')

# Colour palette (matches the mobile app)
ACCENT  = '#22c55e'   # green
ACCENT2 = '#f59e0b'   # amber
ACCENT3 = '#3b82f6'   # blue
DANGER  = '#ef4444'   # red
BG      = '#0a0f1a'
CARD    = '#111827'

plt.rcParams.update({
    'figure.facecolor': BG, 'axes.facecolor': CARD,
    'axes.edgecolor': '#334155', 'text.color': 'white',
    'axes.labelcolor': 'white', 'xtick.color': 'white',
    'ytick.color': 'white', 'grid.color': '#1e293b',
    'grid.linestyle': '--', 'grid.alpha': 0.5, 'font.size': 11,
})

BASE = Path.home() / 'Documents' / 'btc-wheel-bot'

# ── Config ──────────────────────────────────────────────────────────
with open(BASE / 'config.yaml') as f:
    cfg = yaml.safe_load(f)
strat  = cfg.get('strategy', {})
sizing = cfg.get('sizing', {})
back   = cfg.get('backtest', {})

# ── Trade history ────────────────────────────────────────────────────
trades = pd.read_csv(BASE / 'data' / 'backtest_trades.csv',
                     parse_dates=['open_date', 'close_date'])
trades['duration_days']    = (trades['close_date'] - trades['open_date']).dt.days
trades['annualised_yield'] = (
    trades['premium_usd'] / (trades['strike'] * trades['contracts'])
) * (365 / trades['dte'].replace(0, 1)) * 100

# ── Optimiser results ────────────────────────────────────────────────
with open(BASE / 'data' / 'optimizer' / 'monte_carlo_results.json') as f:
    mc_raw = json.load(f)
with open(BASE / 'data' / 'optimizer' / 'sweep_results.json') as f:
    sweep_raw = json.load(f)
with open(BASE / 'data' / 'optimizer' / 'walk_forward_results.json') as f:
    wf_raw = json.load(f)
leaderboard = pd.read_csv(BASE / 'data' / 'optimizer' / 'evolution_leaderboard.csv')

print('✅  All data loaded!')
print(f'   Trades          : {len(trades)}')
print(f'   Monte Carlo runs: {len(mc_raw.get("runs", []))}')
print(f'   Sweep params    : {len(sweep_raw.get("results", {}))}')
print(f'   Leaderboard bots: {len(leaderboard)}')

---
## 2 · The Wheel Strategy — plain English

The **Wheel** is a repeating premium-selling cycle that works particularly well on BTC because of its consistently elevated implied volatility.

```
┌─────────────────────────────────────────────────────────────────────┐
│  STEP 1 — Sell a cash-secured PUT below current BTC price          │
│           Collect premium upfront. Your obligation: buy BTC at     │
│           the strike price if it falls there by expiry.            │
│                                                                     │
│  STEP 2A — BTC stays ABOVE strike → option expires worthless.      │
│            You keep 100% of the premium. Repeat from Step 1.       │
│                                                                     │
│  STEP 2B — BTC falls BELOW strike → you are 'assigned' BTC.       │
│            Cost basis = strike − premium already collected.        │
│                                                                     │
│  STEP 3 — Sell a COVERED CALL at or above your cost basis.         │
│           BTC rises → it's called away at a profit.                │
│           BTC stays flat → collect premium, repeat.                │
└─────────────────────────────────────────────────────────────────────┘
```

### Why it works on BTC specifically

- **IV is overpriced vs realised vol** — option sellers collect more premium than the realised moves justify. This is the core edge.
- **High IV = fat premiums** — the bot only sells when IV Rank is above threshold, ensuring it sells *expensive* options.
- **BTC's long-term upward trend** — assignment just means accumulating BTC at a discount. Not necessarily bad.
- **Systematic execution** — no emotion, no FOMO. The bot follows its rules exactly every time.

### The bot's special ingredients

1. **IV Rank filter** — only sell when implied volatility is historically elevated
2. **Delta targeting** — choose strikes at a specific probability of being ITM
3. **Perpetual futures hedge** — reduce directional exposure after entry
4. **Genetic optimiser** — automatically tunes all parameters to maximise risk-adjusted return
5. **Regime filter** — reduces position size in bear markets

---
## 3 · Options fundamentals — P&L diagrams

Options have non-linear P&L — understanding the shape of the curve is fundamental to everything that follows.

In [ ]:
# ── Black-Scholes put delta ──────────────────────────────────────────
def bs_delta_put(S, K, T, sigma, r=0.05):
    if T <= 0: return -1.0 if S < K else 0.0
    d1 = (math.log(S/K) + (r + 0.5*sigma**2)*T) / (sigma*math.sqrt(T))
    return norm.cdf(d1) - 1

def approx_strike_from_delta(spot, delta_abs, T, sigma, r=0.05):
    """Find the strike where a put has the given absolute delta (e.g. 0.15)"""
    d1_target = norm.ppf(1 - delta_abs)
    log_SK = (d1_target - 0.5*sigma*math.sqrt(T)) * sigma * math.sqrt(T)
    return spot * math.exp(log_SK)

# Representative values
spot    = 85_000
strike  = 80_000
premium = 1_200    # $ per contract-BTC
iv      = 0.80     # 80% IV — typical for BTC
dte     = 21
T       = dte / 365

prices = np.linspace(55_000, 100_000, 500)
pnl    = np.where(prices >= strike, premium, premium - (strike - prices))
deltas = [bs_delta_put(p, strike, T, iv) for p in prices]

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
fig.suptitle('Short Put — Profit & Loss and Delta Curve', fontsize=14, color='white')

# ── P&L diagram ─────────────────────────────────────────────────────
ax = axes[0]
ax.axhline(0, color='#475569', linewidth=0.8)
ax.plot(prices, pnl, color=ACCENT3, linewidth=2.5)
ax.fill_between(prices, pnl, 0, where=np.array(pnl) >= 0, alpha=0.18, color=ACCENT)
ax.fill_between(prices, pnl, 0, where=np.array(pnl) < 0,  alpha=0.18, color=DANGER)
be = strike - premium
ax.axvline(strike, color=ACCENT2, linewidth=1.5, linestyle='--', label=f'Strike ${strike:,.0f}')
ax.axvline(spot,   color='white',  linewidth=1.2, linestyle=':',  label=f'Spot ${spot:,.0f}')
ax.axvline(be,     color=DANGER,   linewidth=1.2, linestyle='--', label=f'Breakeven ${be:,.0f}')
ax.axhline(premium, color=ACCENT,  linewidth=0.8, linestyle=':',  label=f'Max profit ${premium:,.0f}')
ax.annotate('✓ Keep full premium\n  BTC stays above strike',
            xy=(90000, premium * 0.6), color=ACCENT, fontsize=9, ha='center')
ax.annotate('✗ Loss zone\n  (breakeven is ${:,.0f})'.format(be),
            xy=(68000, -4500), color=DANGER, fontsize=9, ha='center')
ax.set_title('Short Put P&L at Expiry', color='white')
ax.set_xlabel('BTC price at expiry')
ax.set_ylabel('P&L (USD)')
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# ── Delta curve ─────────────────────────────────────────────────────
ax = axes[1]
ax.plot(prices, deltas, color=ACCENT3, linewidth=2.5)
ax.axhline(0, color='#475569', linewidth=0.8)
dmin = strat.get('target_delta_min', -0.20)
dmax = strat.get('target_delta_max', -0.10)
ax.axhspan(dmin, dmax, alpha=0.15, color=ACCENT, label=f'Bot target: {dmax} to {dmin}')
ax.axhline(dmin, color=ACCENT, linewidth=1.5, linestyle='--')
ax.axhline(dmax, color=ACCENT, linewidth=1.5, linestyle='--')
ax.axvline(strike, color=ACCENT2, linewidth=1, linestyle=':',  label=f'Example strike ${strike:,.0f}')
ax.axvline(spot,   color='white',  linewidth=1, linestyle=':',  label=f'Spot ${spot:,.0f}')
ax.set_title(f'Put Delta vs BTC Price ({dte} DTE, {iv*100:.0f}% IV)', color='white')
ax.set_xlabel('BTC price')
ax.set_ylabel('Delta')
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.annotate('Bot sells puts\nin this delta band',
            xy=(84000, (dmin+dmax)/2), color=ACCENT, fontsize=9, ha='center')

plt.tight_layout()
plt.savefig(BASE / 'data' / 'chart_options_fundamentals.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print(f'At ${spot:,.0f} spot, ${strike:,.0f} strike, {dte} DTE, {iv*100:.0f}% IV:')
print(f'  Delta = {bs_delta_put(spot, strike, T, iv):.3f}')
print(f'  Breakeven = ${be:,.0f} ({(1-be/spot)*100:.1f}% drop required to lose money)')

### Key takeaways

**Left chart — P&L at expiry:**
- Right of the strike: flat green line. You collect the full premium regardless of where BTC is.
- Left of the breakeven: you lose money. Every dollar BTC drops below the strike costs you a dollar (minus the premium buffer).
- **Maximum profit is known upfront** (the premium). Maximum loss is theoretically the full strike price (BTC goes to zero).

**Right chart — Delta:**
- Delta is a number between −1 and 0 for puts. It tells you: 'if BTC moves $1, how much does my option change in value?'
- Delta −0.15 means: for every $1 BTC moves down, the option gains $0.15. Your position loses $0.15 per $1 BTC drop.
- The green band is where the bot targets its strikes. Far OTM = low delta = lower probability of trouble.
- **Delta also approximates the probability of the option expiring in the money.** Delta −0.15 ≈ 15% chance of loss at expiry.

---
## 4 · Your active config visualised

Every parameter in `config.yaml` has a specific meaning. Let's put them all on a BTC price map.

In [ ]:
# ── Print current config ────────────────────────────────────────────
print('═' * 55)
print('  ACTIVE CONFIG')
print('═' * 55)
print(f"  IV rank threshold : {strat.get('iv_rank_threshold')}")
print(f"    ↳ Only trade when IV percentile rank > this value")
print(f"  Delta target      : {strat.get('target_delta_min')} to {strat.get('target_delta_max')}")
print(f"    ↳ Sell puts at this probability-of-ITM range")
print(f"  OTM offset        : {back.get('approx_otm_offset')} ({float(back.get('approx_otm_offset') or 0)*100:.0f}% below spot)")
print(f"  DTE window        : {strat.get('min_dte')} – {strat.get('max_dte')} days")
print(f"    ↳ Only open trades in this days-to-expiry range")
print(f"  Max equity/leg    : {sizing.get('max_equity_per_leg')} BTC")
print(f"  Min free equity   : {sizing.get('min_free_equity_fraction', 0)*100:.0f}%")
print(f"    ↳ Always keep this fraction of equity unrisked")
print(f"  Premium target    : {float(back.get('premium_fraction_of_spot') or 0)*100:.2f}% of spot")
print(f"  Starting equity   : {back.get('starting_equity')} BTC")
print('═' * 55)

In [ ]:
# ── Visual config map on BTC price ladder ───────────────────────────
spot_price = 85_000
iv_vol     = 0.80
otm_offset = float(back.get('approx_otm_offset', 0.10))
d_min      = float(strat.get('target_delta_min', 0.10))   # conservative (far OTM)
d_max      = float(strat.get('target_delta_max', 0.20))   # aggressive (closer to ATM)
dte_mid    = (int(strat.get('min_dte', 14)) + int(strat.get('max_dte', 30))) / 2 / 365

strike_tight = approx_strike_from_delta(spot_price, d_max, dte_mid, iv_vol)  # closer to spot
strike_wide  = approx_strike_from_delta(spot_price, d_min, dte_mid, iv_vol)  # further from spot
otm_center   = spot_price * (1 - otm_offset)
prem_frac    = float(back.get('premium_fraction_of_spot', 0.006))
prem_usd     = spot_price * prem_frac
be_tight     = strike_tight - prem_usd
be_wide      = strike_wide  - prem_usd

fig, ax = plt.subplots(figsize=(12, 8))

# Main price levels as horizontal lines
ax.axhline(spot_price,    color='white',  linewidth=2.5, label=f'Spot ${spot_price:,.0f}', zorder=6)
ax.axhline(strike_tight,  color=ACCENT,   linewidth=1.5, linestyle='--',
           label=f'Delta {-d_max:.0%} strike ~${strike_tight:,.0f}')
ax.axhline(strike_wide,   color=ACCENT,   linewidth=1.5, linestyle=':',
           label=f'Delta {-d_min:.0%} strike ~${strike_wide:,.0f}')
ax.axhline(otm_center,    color=ACCENT2,  linewidth=2,   linestyle='-.',
           label=f'OTM offset {otm_offset:.0%} = ${otm_center:,.0f}')

# Shaded zones
ax.axhspan(strike_wide, strike_tight, alpha=0.12, color=ACCENT, label='Valid strike zone')
ax.axhspan(be_wide,  be_tight, alpha=0.10, color=DANGER, label='Breakeven zone')

# Annotations
mid_x = 0.75
ax.annotate(f'✓  SAFE ZONE\nBTC is here → keep premium',
            xy=(mid_x, spot_price + 800), xycoords=('axes fraction', 'data'),
            color=ACCENT, fontsize=10, ha='center',
            bbox=dict(boxstyle='round,pad=0.3', fc=CARD, ec=ACCENT, alpha=0.9))
ax.annotate(f'Strike zone\n${strike_wide:,.0f} – ${strike_tight:,.0f}\n(put is sold here)',
            xy=(0.12, (strike_wide+strike_tight)/2), xycoords=('axes fraction', 'data'),
            color=ACCENT, fontsize=9,
            bbox=dict(boxstyle='round,pad=0.3', fc=CARD, ec=ACCENT, alpha=0.8))
ax.annotate(f'⚠  Assignment zone\nBTC here = hold BTC at loss',
            xy=(mid_x, (be_wide+be_tight)/2), xycoords=('axes fraction', 'data'),
            color=DANGER, fontsize=9, ha='center')

ax.set_title(f'Active Config Visualised — BTC ${spot_price:,.0f} | {int(strat.get("min_dte",14))}–{int(strat.get("max_dte",30))} DTE | {iv_vol*100:.0f}% IV',
             color='white', fontsize=13)
ax.set_ylabel('BTC Price (USD)')
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.set_xlim(0, 1)
ax.set_xticks([])
ax.legend(loc='upper right', fontsize=9)
ax.grid(True, axis='y', alpha=0.3)

# DTE window annotation
ax.text(0.02, 0.03,
        f'DTE window: {strat.get("min_dte")}–{strat.get("max_dte")} days  |  Max equity/leg: {sizing.get("max_equity_per_leg")} BTC  |  Premium target: {prem_frac*100:.2f}% of spot = ${prem_usd:,.0f}',
        transform=ax.transAxes, color='#94a3b8', fontsize=9)

plt.tight_layout()
plt.savefig(BASE / 'data' / 'chart_active_config.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

---
## 5 · Short-put mechanics — walking through a real trade

Let's trace the full lifecycle of a real trade from the backtest.

In [ ]:
# Show all trades as a table first
display_cols = ['cycle_num','open_date','close_date','option_type','strike',
                'spot_at_open','dte','iv_rank','premium_usd','pnl_usd','annualised_yield']
print(trades[display_cols].to_string(index=False))

In [ ]:
# ── Pick best and worst trade to analyse ────────────────────────────
t_best  = trades.loc[trades['pnl_usd'].idxmax()]
t_worst = trades.loc[trades['pnl_usd'].idxmin()]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Real Trade Lifecycle — Best vs Worst', fontsize=14, color='white')

for ax, t, label, main_color in [
    (axes[0], t_best,  'BEST TRADE',  ACCENT),
    (axes[1], t_worst, 'WORST TRADE', DANGER),
]:
    strike   = t['strike']
    premium  = t['premium_usd'] / max(t['contracts'], 1)
    spot_o   = t['spot_at_open']
    spot_c   = t['spot_at_close']
    contrs   = t['contracts']

    lo = min(strike * 0.75, spot_c * 0.9)
    hi = max(spot_o * 1.08, spot_c * 1.05)
    prices_t = np.linspace(lo, hi, 400)
    pnl_t = np.where(prices_t >= strike,
                     premium, premium - (strike - prices_t)) * contrs

    ax.axhline(0, color='#475569', linewidth=0.8)
    ax.plot(prices_t, pnl_t, color=main_color, linewidth=2.5, label='P&L curve')
    ax.fill_between(prices_t, pnl_t, 0, where=np.array(pnl_t)>=0, alpha=0.15, color=ACCENT)
    ax.fill_between(prices_t, pnl_t, 0, where=np.array(pnl_t)<0,  alpha=0.15, color=DANGER)
    ax.axvline(strike,  color=ACCENT2, linewidth=1.5, linestyle='--', label=f'Strike ${strike:,.0f}')
    ax.axvline(spot_o,  color='white',  linewidth=1.2, linestyle=':',  label=f'Spot at open ${spot_o:,.0f}')
    ax.axvline(spot_c,  color=main_color, linewidth=2, linestyle='-', label=f'Spot at close ${spot_c:,.0f}')
    be = strike - premium
    ax.axvline(be, color=DANGER, linewidth=1, linestyle='--', alpha=0.7, label=f'Breakeven ${be:,.0f}')

    # Actual outcome dot
    actual = t['pnl_usd']
    ax.scatter([spot_c], [actual], s=180, color=main_color, zorder=10,
               edgecolors='white', linewidth=1.5, label=f'Actual: ${actual:,.0f}')

    ax.set_title(f'{label} — Trade #{int(t["cycle_num"])}\n'
                 f'{t["option_type"]} @ ${strike:,.0f} | {int(t["dte"])} DTE | IV rank {t["iv_rank"]:.0f}',
                 color=main_color, fontsize=10)
    ax.set_xlabel('BTC price at close')
    ax.set_ylabel('Total P&L (USD)')
    ax.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f'${x:,.0f}'))
    ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(BASE / 'data' / 'chart_trade_lifecycle.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

for label, t in [('BEST', t_best), ('WORST', t_worst)]:
    print(f'\n{label} TRADE (#{int(t["cycle_num"])})')
    print(f'  Opened    : {t["open_date"].date()} | Closed: {t["close_date"].date()}')
    print(f'  Type      : {t["option_type"]} @ ${t["strike"]:,.0f}')
    print(f'  BTC open  : ${t["spot_at_open"]:,.0f} | BTC close: ${t["spot_at_close"]:,.0f}')
    print(f'  Premium   : ${t["premium_usd"]:,.0f} | P&L: ${t["pnl_usd"]:,.0f}')
    print(f'  Ann. yield: {t["annualised_yield"]:.0f}%')

---
## 6 · Delta hedging with perpetual futures

A short put position is **directional** — it has negative delta, meaning it loses value when BTC crashes. The bot runs a **perpetual futures short** alongside the options position to partially neutralise this exposure.

### The mechanics

- A **short put** at delta −0.15 means: for every $1 BTC falls, you lose $0.15 per contract-BTC
- A **short perpetual** has delta −1.0: for every $1 BTC falls, the short gains $1
- By shorting 0.15 BTC in perps per contract, you reduce net delta toward zero
- Net position becomes roughly delta-neutral — neither strongly bullish nor bearish

### Why not fully hedge?

- Fully hedging kills the premium income advantage (the edge comes from being short volatility, not direction)
- Perp shorts pay/receive **funding rate** which can be significant
- The hedge is adjusted as delta changes (rebalancing), creating ongoing management costs
- The bot sizes the hedge based on the option's current delta and rebalances when it drifts

In [ ]:
# ── Visualise hedged vs unhedged P&L ───────────────────────────────
spot_e = 86_000
strike_h = 80_000
premium_h = 1_400
option_delta = -0.15   # put delta at entry
hedge_btc = abs(option_delta)   # BTC shorted in perps

prices_h = np.linspace(60_000, 100_000, 400)

# Unhedged: short put only
put_pnl = np.where(prices_h >= strike_h,
                   premium_h,
                   premium_h - (strike_h - prices_h))

# Perp P&L: short hedge_btc BTC from spot_e
perp_pnl = hedge_btc * (spot_e - prices_h)   # profits when price falls

# Combined
hedged_pnl = put_pnl + perp_pnl

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Delta Hedging — Reducing Directional Risk', fontsize=14, color='white')

# ── P&L comparison ──────────────────────────────────────────────────
ax = axes[0]
ax.axhline(0, color='#475569', linewidth=0.8)
ax.plot(prices_h, put_pnl,    color=DANGER,  linewidth=2,   linestyle='--', label='Short put only (unhedged)', alpha=0.8)
ax.plot(prices_h, perp_pnl,   color=ACCENT3, linewidth=1.5, linestyle=':',  label=f'Perp short ({hedge_btc:.2f} BTC)', alpha=0.8)
ax.plot(prices_h, hedged_pnl, color=ACCENT,  linewidth=2.5, label='Net hedged P&L')
ax.fill_between(prices_h, hedged_pnl, 0, where=np.array(hedged_pnl)>=0, alpha=0.12, color=ACCENT)
ax.fill_between(prices_h, hedged_pnl, 0, where=np.array(hedged_pnl)<0,  alpha=0.12, color=DANGER)
ax.axvline(strike_h, color=ACCENT2, linewidth=1, linestyle='--', label=f'Strike ${strike_h:,.0f}')
ax.axvline(spot_e,   color='white',  linewidth=1, linestyle=':',  label=f'Entry ${spot_e:,.0f}')
ax.set_title('P&L: Hedged vs Unhedged', color='white')
ax.set_xlabel('BTC price at expiry')
ax.set_ylabel('P&L (USD)')
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# ── Net delta at each price ──────────────────────────────────────────
ax2 = axes[1]
T_h = 21/365
net_deltas = [bs_delta_put(p, strike_h, T_h, 0.80) + (-hedge_btc) * (-1) for p in prices_h]
# Note: short perp adds -1 delta per BTC (bearish). Short put adds negative delta.
# Net delta = put_delta + (-hedge_btc)  [perp short = negative delta]
net_deltas_correct = [bs_delta_put(p, strike_h, T_h, 0.80) - hedge_btc for p in prices_h]
put_deltas = [bs_delta_put(p, strike_h, T_h, 0.80) for p in prices_h]

ax2.plot(prices_h, put_deltas,         color=DANGER,  linewidth=2,   linestyle='--', label='Put delta (unhedged)', alpha=0.8)
ax2.plot(prices_h, [-hedge_btc]*len(prices_h), color=ACCENT3, linewidth=1.5, linestyle=':', label=f'Perp delta (−{hedge_btc:.2f})', alpha=0.8)
ax2.plot(prices_h, net_deltas_correct, color=ACCENT,  linewidth=2.5, label='Net delta')
ax2.axhline(0, color='white', linewidth=1.5, linestyle='--', alpha=0.5, label='Delta = 0 (neutral)')
ax2.axvline(strike_h, color=ACCENT2, linewidth=1, linestyle='--')
ax2.axvline(spot_e,   color='white',  linewidth=1, linestyle=':')
ax2.set_title('Delta Profile: Hedged vs Unhedged', color='white')
ax2.set_xlabel('BTC price')
ax2.set_ylabel('Delta')
ax2.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(BASE / 'data' / 'chart_delta_hedge.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

# Show impact at specific scenarios
for scenario, price in [('BTC −5%', spot_e*0.95), ('BTC −15%', spot_e*0.85), ('BTC −30%', spot_e*0.70)]:
    put = float(np.interp(price, prices_h, put_pnl))
    hedged = float(np.interp(price, prices_h, hedged_pnl))
    improvement = hedged - put
    print(f'{scenario}: Unhedged ${put:>8,.0f}  |  Hedged ${hedged:>8,.0f}  |  Hedge benefit ${improvement:>+8,.0f}')

---
## 7 · Backtest trade history — full analysis of all 27 trades

In [ ]:
wins   = (trades['pnl_usd'] > 0).sum()
losses = (trades['pnl_usd'] <= 0).sum()

fig = plt.figure(figsize=(16, 13))
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

# ── Equity curve ────────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, :])
eq = trades.sort_values('close_date')
ax1.plot(eq['close_date'], eq['equity_after'], color=ACCENT, linewidth=2,
         marker='o', markersize=5, zorder=5)
ax1.fill_between(eq['close_date'],
                 float(eq['equity_after'].min()) * 0.995,
                 eq['equity_after'], alpha=0.15, color=ACCENT)
ax1.set_title('Equity Curve (BTC) — Every Vertical Line = a Closed Trade', color='white', fontsize=11)
ax1.set_ylabel('Equity (BTC)')
ax1.grid(True, alpha=0.3)
for _, row in eq.iterrows():
    ax1.axvline(row['close_date'],
                color=ACCENT if row['pnl_usd'] >= 0 else DANGER,
                alpha=0.25, linewidth=1.2)

# ── P&L distribution ────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[1, 0])
ax2.hist(trades['pnl_usd'], bins=12, color=ACCENT, alpha=0.8, edgecolor=BG, linewidth=0.5)
ax2.axvline(trades['pnl_usd'].mean(), color=ACCENT2, linewidth=2,
            label=f'Mean ${trades["pnl_usd"].mean():,.0f}')
ax2.axvline(0, color='white', linewidth=1, linestyle='--')
ax2.set_title('P&L Distribution (USD)', color='white')
ax2.set_xlabel('P&L per trade ($)')
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

# ── Win/loss pie ────────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 1])
ax3.pie([wins, losses], labels=[f'Wins ({wins})', f'Losses ({losses})'],
        colors=[ACCENT, DANGER], autopct='%1.0f%%', startangle=90,
        textprops={'color': 'white'},
        wedgeprops={'edgecolor': BG, 'linewidth': 2})
ax3.set_title(f'Win Rate — {wins}/{len(trades)} trades profitable', color='white')

# ── Annualised yield ────────────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 2])
ax4.hist(trades['annualised_yield'], bins=12, color=ACCENT3, alpha=0.8, edgecolor=BG, linewidth=0.5)
ax4.axvline(trades['annualised_yield'].median(), color=ACCENT2, linewidth=2,
            label=f'Median {trades["annualised_yield"].median():.0f}%/yr')
ax4.set_title('Annualised Yield % per Trade', color='white')
ax4.set_xlabel('Annualised yield (%)')
ax4.legend(fontsize=8)
ax4.grid(True, alpha=0.3)

# ── DTE vs P&L scatter ──────────────────────────────────────────────
ax5 = fig.add_subplot(gs[2, 0])
c5 = [ACCENT if p > 0 else DANGER for p in trades['pnl_usd']]
ax5.scatter(trades['dte'], trades['pnl_usd'], c=c5, s=70, alpha=0.85,
            edgecolors='white', linewidth=0.4)
ax5.axhline(0, color='#475569', linewidth=0.8)
ax5.set_title('DTE at Entry vs P&L', color='white')
ax5.set_xlabel('Days to expiry at entry')
ax5.set_ylabel('P&L ($)')
ax5.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax5.grid(True, alpha=0.3)

# ── IV rank vs premium ──────────────────────────────────────────────
ax6 = fig.add_subplot(gs[2, 1])
sc6 = ax6.scatter(trades['iv_rank'], trades['premium_usd'],
                  c=trades['pnl_usd'], cmap='RdYlGn',
                  s=80, alpha=0.9, edgecolors='white', linewidth=0.3)
plt.colorbar(sc6, ax=ax6, label='P&L ($)')
ax6.set_title('IV Rank vs Premium Collected', color='white')
ax6.set_xlabel('IV Rank at entry')
ax6.set_ylabel('Premium ($)')
ax6.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax6.grid(True, alpha=0.3)

# ── Cumulative P&L ──────────────────────────────────────────────────
ax7 = fig.add_subplot(gs[2, 2])
eq_s = trades.sort_values('close_date').copy()
eq_s['cum_pnl'] = eq_s['pnl_usd'].cumsum()
colors_cum = [ACCENT if v >= 0 else DANGER for v in eq_s['cum_pnl']]
ax7.bar(range(len(eq_s)), eq_s['pnl_usd'], color=colors_cum, alpha=0.8, edgecolor=BG)
ax7_twin = ax7.twinx()
ax7_twin.plot(range(len(eq_s)), eq_s['cum_pnl'], color=ACCENT2, linewidth=2, label='Cumulative')
ax7_twin.set_ylabel('Cumulative P&L ($)', color=ACCENT2)
ax7_twin.tick_params(axis='y', colors=ACCENT2)
ax7.axhline(0, color='white', linewidth=0.8)
ax7.set_title('Per-Trade P&L + Cumulative', color='white')
ax7.set_xlabel('Trade number')
ax7.set_ylabel('Per-trade P&L ($)')
ax7.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax7_twin.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax7.grid(True, axis='y', alpha=0.3)

plt.suptitle('Complete Backtest Trade Analysis — 27 Trades', fontsize=15, color='white', y=1.01)
plt.savefig(BASE / 'data' / 'chart_backtest_analysis.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

print('═' * 55)
print('  TRADE STATISTICS')
print('═' * 55)
print(f'  Total trades      : {len(trades)}')
print(f'  Win rate          : {wins}/{len(trades)} = {wins/len(trades)*100:.0f}%')
print(f'  Total P&L (USD)   : ${trades["pnl_usd"].sum():>10,.0f}')
print(f'  Average P&L       : ${trades["pnl_usd"].mean():>10,.0f}')
print(f'  Best trade        : ${trades["pnl_usd"].max():>10,.0f}')
print(f'  Worst trade       : ${trades["pnl_usd"].min():>10,.0f}')
print(f'  Avg premium/trade : ${trades["premium_usd"].mean():>10,.0f}')
print(f'  Avg DTE at entry  : {trades["dte"].mean():>8.0f} days')
print(f'  Avg duration      : {trades["duration_days"].mean():>8.1f} days')
print(f'  Median ann. yield : {trades["annualised_yield"].median():>8.0f}%')
print('═' * 55)

---
## 8 · Parameter sweep — sensitivity curves

The optimiser tested each parameter individually, sweeping it across a range while holding everything else fixed. The curves show you **how sensitive the bot's performance is to each parameter** — steep curves mean a parameter matters a lot; flat curves mean it's less important.

**Fitness score** is a composite metric: higher = better. It blends return, Sharpe ratio, win rate, and drawdown.

In [ ]:
sweep = sweep_raw   # top-level keys ARE the param names

param_labels = {
    'iv_rank_threshold':        'IV Rank Threshold',
    'target_delta_min':         'Delta Min (abs)',
    'target_delta_max':         'Delta Max (abs)',
    'approx_otm_offset':        'OTM Offset',
    'max_dte':                  'Max DTE',
    'min_dte':                  'Min DTE',
    'max_equity_per_leg':       'Max Equity/Leg (BTC)',
    'premium_fraction_of_spot': 'Premium Fraction of Spot',
    'iv_rank_window_days':      'IV Rank Window (days)',
    'min_free_equity_fraction': 'Min Free Equity Fraction',
    'starting_equity':          'Starting Equity',
}

params_found = [p for p in param_labels if p in sweep]
n    = len(params_found)
cols = 3
rows = max(1, math.ceil(n / cols))

fig, axes = plt.subplots(rows, cols, figsize=(14, rows * 3.8))
fig.suptitle('Parameter Sweep — How Each Config Knob Affects Performance', fontsize=14, color='white')
axes_flat = axes.flatten() if hasattr(axes, 'flatten') else [axes]

for i, param in enumerate(params_found):
    ax = axes_flat[i]
    entries = sweep[param]
    # x = the value of this parameter in each sweep entry
    xs      = [e['params'][param]       for e in entries]
    fitness = [e['fitness']             for e in entries]
    returns = [e.get('total_return_pct', e.get('return_pct', 0)) for e in entries]

    ax.plot(xs, fitness, color=ACCENT, linewidth=2.5, marker='o', ms=5, label='Fitness')

    ax2 = ax.twinx()
    ax2.plot(xs, returns, color=ACCENT3, linewidth=1.5, linestyle='--',
             marker='s', ms=4, alpha=0.7, label='Return %')
    ax2.tick_params(axis='y', colors=ACCENT3, labelsize=8)
    ax2.set_ylabel('Return %', color=ACCENT3, fontsize=8)

    best_idx = fitness.index(max(fitness))
    ax.axvline(xs[best_idx], color=ACCENT2, linewidth=1.5, linestyle='--',
               label=f'Best: {xs[best_idx]:.4g}')

    # Mark current config value
    cfg_map = {
        'iv_rank_threshold':        strat.get('iv_rank_threshold'),
        'target_delta_min':         strat.get('target_delta_min'),
        'target_delta_max':         strat.get('target_delta_max'),
        'approx_otm_offset':        back.get('approx_otm_offset'),
        'max_dte':                  strat.get('max_dte'),
        'min_dte':                  strat.get('min_dte'),
        'max_equity_per_leg':       sizing.get('max_equity_per_leg'),
        'premium_fraction_of_spot': back.get('premium_fraction_of_spot'),
        'min_free_equity_fraction': sizing.get('min_free_equity_fraction'),
        'starting_equity':          back.get('starting_equity'),
    }
    current_val = cfg_map.get(param)
    if current_val is not None:
        ax.axvline(float(current_val), color='white', linewidth=1.2, linestyle=':',
                   alpha=0.6, label=f'Current: {float(current_val):.4g}')

    ax.set_title(param_labels.get(param, param), color='white', fontsize=10)
    ax.set_xlabel('Value', fontsize=8)
    ax.set_ylabel('Fitness', fontsize=8, color=ACCENT)
    ax.tick_params(axis='y', colors=ACCENT, labelsize=8)
    ax.legend(fontsize=7, loc='lower right')
    ax.grid(True, alpha=0.25)

for j in range(n, len(axes_flat)):
    axes_flat[j].set_visible(False)

plt.tight_layout()
plt.savefig(BASE / 'data' / 'chart_sweep_sensitivity.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

print('=== BEST VALUE PER PARAMETER ===')
for param in params_found:
    entries = sweep[param]
    fitness_vals = [e['fitness'] for e in entries]
    best = entries[fitness_vals.index(max(fitness_vals))]
    best_val = best['params'][param]
    curr = cfg_map.get(param)  # None if not in config
    match = '✅' if curr is not None and abs(float(str(curr)) - float(str(best_val))) < abs(float(str(best_val)))*0.05 + 0.001 else ('⚠️' if curr is None else '❓')
    curr_str = f'{float(str(curr)):.4g}' if curr is not None else 'N/A'
    print(f'  {match} {param_labels.get(param,param):<35}: best={best_val:>8.4g}  current={curr_str:>8}  fitness={max(fitness_vals):.3f}')


---
## 9 · Genetic algorithm — the evolution of the best bot

The genetic optimiser runs hundreds of 'bots' (each with randomly varied config parameters), measures their backtest performance, then combines the best performers to create the next generation. Over many generations, the population converges on near-optimal parameter sets.

Think of it like natural selection: survival of the fittest bot.

In [ ]:
lb = leaderboard.copy()
print(f'Leaderboard: {len(lb)} bots')

fig, axes = plt.subplots(2, 2, figsize=(14, 11))
fig.suptitle('Genetic Evolution — Leaderboard Analysis', fontsize=14, color='white')

# ── Return vs Sharpe ────────────────────────────────────────────────
ax = axes[0, 0]
sc = ax.scatter(lb['total_return_pct'], lb['sharpe_ratio'],
                c=lb['fitness'], cmap='viridis',
                s=50, alpha=0.75, edgecolors='white', linewidth=0.2)
plt.colorbar(sc, ax=ax, label='Fitness')
top5 = lb.nlargest(5, 'fitness')
ax.scatter(top5['total_return_pct'], top5['sharpe_ratio'],
           s=180, color=ACCENT2, zorder=10, edgecolors='white',
           linewidth=1.5, label='Top 5 bots')
for idx, (_, row) in enumerate(top5.iterrows()):
    ax.annotate(f'#{idx+1}', (row['total_return_pct'], row['sharpe_ratio']),
                textcoords='offset points', xytext=(6, 3), fontsize=8, color='white')
ax.set_title('Return vs Sharpe (coloured by fitness)', color='white')
ax.set_xlabel('Total Return %')
ax.set_ylabel('Sharpe Ratio')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# ── Fitness distribution ─────────────────────────────────────────────
ax = axes[0, 1]
ax.hist(lb['fitness'], bins=20, color=ACCENT3, alpha=0.85, edgecolor=BG, linewidth=0.4)
p90 = lb['fitness'].quantile(0.90)
ax.axvline(p90, color=ACCENT2, linewidth=2, label=f'Top 10%: {p90:.2f}')
ax.axvline(lb['fitness'].max(), color=ACCENT, linewidth=2,
           label=f'Best: {lb["fitness"].max():.2f}')
ax.set_title('Fitness Score Distribution', color='white')
ax.set_xlabel('Fitness')
ax.set_ylabel('Number of bots')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# ── Win rate vs drawdown ─────────────────────────────────────────────
ax = axes[1, 0]
is_top = lb['fitness'] >= p90
ax.scatter(lb.loc[~is_top, 'max_drawdown_pct'], lb.loc[~is_top, 'win_rate_pct'],
           color=ACCENT3, s=40, alpha=0.5, edgecolors='none', label='Rest of population')
ax.scatter(lb.loc[is_top, 'max_drawdown_pct'], lb.loc[is_top, 'win_rate_pct'],
           color=ACCENT2, s=80, alpha=0.9, edgecolors='white', linewidth=0.5, label='Top 10%')
ax.set_title('Win Rate vs Max Drawdown', color='white')
ax.set_xlabel('Max Drawdown %')
ax.set_ylabel('Win Rate %')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# ── Generation convergence (if data exists) ──────────────────────────
ax = axes[1, 1]
if 'generation' in lb.columns and lb['generation'].nunique() > 1:
    gen_stats = lb.groupby('generation').agg(
        best_fitness=('fitness','max'),
        mean_fitness=('fitness','mean'),
    ).reset_index()
    ax.plot(gen_stats['generation'], gen_stats['best_fitness'],
            color=ACCENT, linewidth=2.5, label='Best fitness')
    ax.fill_between(gen_stats['generation'],
                    gen_stats['mean_fitness'], gen_stats['best_fitness'],
                    alpha=0.15, color=ACCENT)
    ax.plot(gen_stats['generation'], gen_stats['mean_fitness'],
            color=ACCENT3, linewidth=1.5, linestyle='--', label='Mean fitness')
    ax.set_title('Fitness Convergence Over Generations', color='white')
    ax.set_xlabel('Generation')
    ax.set_ylabel('Fitness')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
else:
    # Show top bot stats as a table instead
    col_show = [c for c in ['fitness','total_return_pct','sharpe_ratio',
                             'win_rate_pct','max_drawdown_pct','num_cycles'] if c in lb.columns]
    top3 = lb.nlargest(3, 'fitness')[col_show]
    ax.axis('off')
    t = ax.table(cellText=top3.round(2).values,
                 colLabels=(['Fitness','Return%','Sharpe','Win%','Drawdown%','Cycles'] + ['']*(10))[:len(col_show)],
                 loc='center', cellLoc='center')
    t.auto_set_font_size(False)
    t.set_fontsize(9)
    for key, cell in t.get_celld().items():
        cell.set_facecolor(CARD if key[0] > 0 else '#1e3a5f')
        cell.set_text_props(color='white')
        cell.set_edgecolor('#334155')
    ax.set_title('Top 3 Bots from Evolution', color='white')

plt.tight_layout()
plt.savefig(BASE / 'data' / 'chart_evolution.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

print('=== TOP 5 BOTS ===')
col_s = [c for c in ['fitness','total_return_pct','sharpe_ratio','win_rate_pct','max_drawdown_pct','num_cycles'] if c in lb.columns]
print(lb.nlargest(5, 'fitness')[col_s].round(3).to_string())

---
## 10 · Monte Carlo simulation — 100 possible futures

Monte Carlo answers: **'Is my backtest result luck or skill?'**

It takes the trade returns from the backtest, shuffles their order randomly 100 times, and runs each shuffle as a separate 'simulation'. This shows the full range of outcomes the strategy could have produced — not just the one historical path that happened.

- **Probability of profit** tells you: across all 100 simulations, what fraction ended above zero?
- A high probability (>80%) means the edge is real and robust to luck.
- A low probability (<50%) means the original backtest was lucky.

In [ ]:
runs = mc_raw.get('runs', [])
prob_profit = mc_raw.get('prob_profit_pct', None)

returns_mc   = [r.get('return_pct', 0)   for r in runs]
sharpes_mc   = [r.get('sharpe', 0)        for r in runs]
drawdowns_mc = [r.get('max_drawdown', 0)  for r in runs]
win_rates_mc = [r.get('win_rate', 0)      for r in runs]
fitnesses_mc = [r.get('fitness', 0)       for r in runs]

if prob_profit is None:
    prob_profit = sum(r > 0 for r in returns_mc) / len(returns_mc) * 100

fig, axes = plt.subplots(2, 2, figsize=(14, 11))
fig.suptitle(f'Monte Carlo — {len(runs)} Simulated Runs  |  P(Profit) = {prob_profit:.0f}%',
             fontsize=14, color='white')

# ── Return histogram ─────────────────────────────────────────────────
ax = axes[0, 0]
ax.hist(returns_mc, bins=20, color=ACCENT, alpha=0.85, edgecolor=BG, linewidth=0.3)
ax.axvline(0, color=DANGER, linewidth=2, linestyle='--', label='Break-even')
ax.axvline(float(np.mean(returns_mc)), color=ACCENT2, linewidth=2,
           label=f'Mean {np.mean(returns_mc):.1f}%')
ax.axvline(float(np.percentile(returns_mc, 5)), color='#94a3b8', linewidth=1.5,
           linestyle=':', label=f'Worst 5%: {np.percentile(returns_mc, 5):.1f}%')
ax.axvline(float(np.percentile(returns_mc, 95)), color='#94a3b8', linewidth=1.5,
           linestyle='-.', label=f'Best 5%: {np.percentile(returns_mc, 95):.1f}%')
ax.set_title('Return % Distribution', color='white')
ax.set_xlabel('Total Return %')
ax.set_ylabel('Simulations')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# ── Sharpe distribution ──────────────────────────────────────────────
ax = axes[0, 1]
ax.hist(sharpes_mc, bins=20, color=ACCENT3, alpha=0.85, edgecolor=BG, linewidth=0.3)
ax.axvline(0, color=DANGER, linewidth=1.5, linestyle='--', label='Sharpe = 0')
ax.axvline(1, color=ACCENT2, linewidth=1.5, linestyle='--', label='Sharpe = 1 (good)')
ax.axvline(float(np.mean(sharpes_mc)), color=ACCENT, linewidth=2,
           label=f'Mean {np.mean(sharpes_mc):.2f}')
ax.set_title('Sharpe Ratio Distribution', color='white')
ax.set_xlabel('Sharpe Ratio')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# ── Drawdown vs Return scatter ───────────────────────────────────────
ax = axes[1, 0]
sc = ax.scatter(drawdowns_mc, returns_mc,
                c=fitnesses_mc, cmap='RdYlGn', s=60, alpha=0.8,
                edgecolors='white', linewidth=0.2)
plt.colorbar(sc, ax=ax, label='Fitness')
ax.axhline(0, color=DANGER, linewidth=1, linestyle='--')
ax.set_title('Drawdown vs Return (coloured by fitness)', color='white')
ax.set_xlabel('Max Drawdown %')
ax.set_ylabel('Return %')
ax.grid(True, alpha=0.3)

# ── Cumulative probability of return ────────────────────────────────
ax = axes[1, 1]
sorted_ret = sorted(returns_mc)
cumprob = np.arange(1, len(sorted_ret)+1) / len(sorted_ret) * 100
ax.plot(sorted_ret, cumprob, color=ACCENT3, linewidth=2.5)
ax.axvline(0, color=DANGER, linewidth=1.5, linestyle='--', label='Break-even')
loss_prob = 100 - prob_profit
ax.axhline(loss_prob, color=DANGER, linewidth=1, linestyle=':', alpha=0.6,
           label=f'{loss_prob:.0f}% chance of loss')
ax.axhline(prob_profit, color=ACCENT, linewidth=1, linestyle=':',
           label=f'{prob_profit:.0f}% chance of profit')
ax.fill_betweenx(cumprob, sorted_ret, 0,
                 where=np.array(sorted_ret) < 0, alpha=0.15, color=DANGER)
ax.fill_betweenx(cumprob, sorted_ret, 0,
                 where=np.array(sorted_ret) >= 0, alpha=0.15, color=ACCENT)
ax.set_title('Cumulative Probability of Return', color='white')
ax.set_xlabel('Return %')
ax.set_ylabel('Probability of being AT OR BELOW (%)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(BASE / 'data' / 'chart_monte_carlo.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

print('═' * 55)
print('  MONTE CARLO SUMMARY')
print('═' * 55)
print(f'  Runs              : {len(runs)}')
print(f'  P(Profit)         : {prob_profit:.0f}%')
print(f'  Mean return       : {np.mean(returns_mc):.1f}%')
print(f'  Median return     : {np.median(returns_mc):.1f}%')
print(f'  Worst 5% return   : {np.percentile(returns_mc, 5):.1f}%')
print(f'  Best 5% return    : {np.percentile(returns_mc, 95):.1f}%')
print(f'  Mean Sharpe       : {np.mean(sharpes_mc):.2f}')
print(f'  Pct with Sharpe>1 : {sum(s>1 for s in sharpes_mc)/len(sharpes_mc)*100:.0f}%')
print(f'  Mean max drawdown : {np.mean(drawdowns_mc):.1f}%')
print('═' * 55)

---
## 11 · Walk-forward validation — does it actually generalise?

**The gold standard test.** Any strategy can be curve-fitted to look great on historical data. Walk-forward validation checks whether the optimised parameters produce positive results on data the optimiser never saw.

**Process:**
1. Take the full dataset. Split 75/25: first 75% = in-sample (training), last 25% = out-of-sample (test).
2. Optimise all parameters on the training set only.
3. Freeze those parameters and run them cold on the test set.
4. Compare: if OOS ≈ IS → the edge is real. If OOS degrades severely → overfitting.

In [ ]:
is_data  = wf_raw.get('in_sample',      {})
oos_data = wf_raw.get('out_of_sample',  {})
robust   = wf_raw.get('robustness_score', None)

is_sharpe  = is_data.get('sharpe', 0)
oos_sharpe = oos_data.get('sharpe', 0)
is_ret     = is_data.get('return_pct', 0)
oos_ret    = oos_data.get('return_pct', 0)
is_days    = is_data.get('days', 555)
oos_days   = oos_data.get('days', 185)

fig, axes = plt.subplots(1, 3, figsize=(15, 7))
fig.suptitle('Walk-Forward Validation', fontsize=14, color='white')

# ── Bar comparison ───────────────────────────────────────────────────
ax = axes[0]
metrics = ['Sharpe', 'Return %']
is_vals  = [is_sharpe, is_ret]
oos_vals = [oos_sharpe, oos_ret]
x = np.arange(len(metrics))
w = 0.35
b1 = ax.bar(x - w/2, is_vals,  w, label=f'In-sample ({is_days}d)',     color=ACCENT3, alpha=0.85, edgecolor=BG)
b2 = ax.bar(x + w/2, oos_vals, w, label=f'Out-of-sample ({oos_days}d)', color=ACCENT,  alpha=0.85, edgecolor=BG)
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.axhline(0, color='#475569', linewidth=0.8)
ax.legend(fontsize=9)
ax.set_title('IS vs OOS: Key Metrics', color='white')
ax.grid(True, axis='y', alpha=0.3)
for bar in list(b1) + list(b2):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + (0.02 if h >= 0 else -0.08),
            f'{h:.2f}', ha='center', fontsize=9, color='white')

# ── Timeline split ───────────────────────────────────────────────────
ax = axes[1]
total = is_days + oos_days
ax.barh([0], [is_days],  height=0.5, color=ACCENT3, alpha=0.85, label=f'In-sample ({is_days}d — TRAINING)')
ax.barh([0], [oos_days], left=is_days, height=0.5, color=ACCENT, alpha=0.85, label=f'Out-of-sample ({oos_days}d — TEST)')
ax.axvline(is_days, color='white', linewidth=2.5, linestyle='--')
ax.annotate('Optimiser\nstops here →', xy=(is_days, 0.35),
            xytext=(is_days * 0.7, 0.6),
            color='white', fontsize=9, ha='center',
            arrowprops=dict(arrowstyle='->', color='white', lw=1.2))
ax.set_yticks([])
ax.set_xlabel('Days from start')
ax.set_title('Data Split — Optimiser Never Sees the Right Side', color='white')
ax.legend(fontsize=9, loc='lower right')
ax.grid(True, axis='x', alpha=0.3)

# ── Robustness gauge ─────────────────────────────────────────────────
ax = axes[2]
if robust is not None:
    gauge = min(max(robust, 0), 2.5)
    color = ACCENT if robust >= 1.0 else ACCENT2 if robust >= 0.5 else DANGER
    if robust >= 1.5:  interpretation = 'Exceptional ✅'
    elif robust >= 1.0: interpretation = 'Excellent ✅'
    elif robust >= 0.5: interpretation = 'Moderate ⚠️'
    else:               interpretation = 'Poor ❌'
    ax.barh([0], [gauge], height=0.45, color=color, alpha=0.9, edgecolor='white')
    ax.axvline(1.0, color='white', linewidth=2.5, linestyle='--')
    ax.text(gauge/2, 0, f'{robust:.2f}', ha='center', va='center',
            color='white', fontsize=18, fontweight='bold')
    ax.set_xlim(0, 2.5)
    ax.set_yticks([])
    ax.set_xlabel('Robustness Score (OOS Sharpe / IS Sharpe)')
    ax.set_title(f'Robustness: {robust:.2f} — {interpretation}', color=color, fontsize=11)
    ax.text(0.5, -0.18,
            '>1.0 = OOS beat IS (strategy improves on unseen data)',
            ha='center', transform=ax.transAxes, color='#94a3b8', fontsize=9)
    ax.grid(True, axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(BASE / 'data' / 'chart_walk_forward.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

print('═' * 55)
print('  WALK-FORWARD RESULTS')
print('═' * 55)
print(f'  In-sample  ({is_days} days) : Sharpe {is_sharpe:.2f} | Return {is_ret:.1f}% | Cycles {is_data.get("num_cycles", "N/A")}')
print(f'  Out-of-sample ({oos_days}d) : Sharpe {oos_sharpe:.2f} | Return {oos_ret:.1f}% | Cycles {oos_data.get("num_cycles", "N/A")}')
if robust is not None:
    print(f'  Robustness score : {robust:.4f} ({interpretation})')
print('═' * 55)

---
## 12 · Black swan stress tests

BTC has crashed 30%, 40%, even 80% in previous cycles. What happens to the bot's position under these scenarios?

**Important:** The loss is bounded by the **strike price** (not the spot price), and reduced by the **premium you already collected**. The delta hedge further reduces losses.

In [ ]:
spot_e2   = 85_000
otm       = float(back.get('approx_otm_offset', 0.10))
strike_e2 = spot_e2 * (1 - otm)
prem_frac2 = float(back.get('premium_fraction_of_spot', 0.006))
prem_e2   = spot_e2 * prem_frac2
leg_btc   = float(sizing.get('max_equity_per_leg', 0.5))
equity_usd2 = float(back.get('starting_equity', 10000))  # USD
delta_h   = float(strat.get('target_delta_min', 0.10))

scenarios = {
    'BTC +5%   (minor rally)':     spot_e2 * 1.05,
    'BTC flat  (stays put)':       spot_e2 * 1.00,
    'BTC -5%   (normal dip)':      spot_e2 * 0.95,
    'BTC -10%  (moderate drop)':   spot_e2 * 0.90,
    'BTC -20%  (hard drop)':       spot_e2 * 0.80,
    'BTC -30%  (severe crash)':    spot_e2 * 0.70,
    'BTC -40%  (black swan)':      spot_e2 * 0.60,
    'BTC -60%  (bear market)':     spot_e2 * 0.40,
}

results_stress = []
for name, price in scenarios.items():
    if price >= strike_e2:
        put_pnl_btc = prem_e2
    else:
        put_pnl_btc = prem_e2 - (strike_e2 - price)
    total_put    = put_pnl_btc * leg_btc
    hedge_pnl    = delta_h * leg_btc * (spot_e2 - price)
    total_hedged = total_put + hedge_pnl
    pct_eq       = total_put / max(equity_usd2, 1) * 100
    pct_hedged   = total_hedged / max(equity_usd2, 1) * 100
    results_stress.append({
        'scenario': name, 'price': price,
        'put_pnl': total_put, 'hedged_pnl': total_hedged,
        'pct_eq': pct_eq, 'pct_hedged': pct_hedged,
    })

df_s = pd.DataFrame(results_stress)

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
fig.suptitle('Black Swan Stress Test', fontsize=14, color='white')

ax = axes[0]
y  = np.arange(len(df_s))
w  = 0.38
c_put    = [ACCENT if v >= 0 else DANGER for v in df_s['put_pnl']]
c_hedged = [ACCENT if v >= 0 else ACCENT2 for v in df_s['hedged_pnl']]
ax.barh(y + w/2, df_s['put_pnl'],    w, color=c_put,    alpha=0.8, label='Unhedged put P&L')
ax.barh(y - w/2, df_s['hedged_pnl'], w, color=c_hedged, alpha=0.8, label='Hedged P&L')
ax.set_yticks(y)
ax.set_yticklabels(df_s['scenario'], fontsize=9)
ax.axvline(0, color='white', linewidth=1.5)
ax.set_xlabel('P&L (USD)')
ax.set_title('P&L per Scenario: Unhedged vs Hedged', color='white')
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend(fontsize=9)
ax.grid(True, axis='x', alpha=0.3)

ax = axes[1]
c_eq = [ACCENT if v >= 0 else DANGER for v in df_s['pct_eq']]
ax.barh(y, df_s['pct_eq'], color=c_eq, alpha=0.85, edgecolor=BG)
ax.axvline(0, color='white', linewidth=1.5)
for idx, row in df_s.iterrows():
    v = row['pct_eq']
    ax.text(v + (0.05 if v >= 0 else -0.05), idx,
            f'{v:.1f}%', va='center', ha='left' if v >= 0 else 'right',
            fontsize=8, color='white')
ax.set_yticks(y)
ax.set_yticklabels(df_s['scenario'], fontsize=9)
ax.set_xlabel('P&L as % of total equity')
ax.set_title('Impact as % of Total Equity', color='white')
ax.grid(True, axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(BASE / 'data' / 'chart_black_swan.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

print(f'Strike: ${strike_e2:,.0f} | Premium: ${prem_e2:,.2f} | Leg: {leg_btc:.3f} BTC | Equity: ${equity_usd2:,.0f}')
be = strike_e2 - prem_e2
print(f'Breakeven: ${be:,.0f}  (BTC must fall {(1-be/spot_e2)*100:.1f}% to lose money)')
print()
print(f'{"Scenario":<30} {"Put P&L":>12} {"Hedged P&L":>13} {"Equity %":>10}')
print('-' * 70)
for _, row in df_s.iterrows():
    print(f'{row["scenario"]:<30} ${row["put_pnl"]:>10,.0f}  ${row["hedged_pnl"]:>10,.0f}  {row["pct_eq"]:>8.1f}%')


---
## 13 · Strategy improvement ideas — data-driven

Based on everything above — the backtest, sweep results, evolution leaderboard, and walk-forward validation — here are the highest-leverage improvements available.

In [ ]:
improvements = [
    dict(rank=1, effort=2, impact=9, color='#22c55e',
         title='Increase trade frequency',
         detail='Only 27 trades over the full backtest period. More cycles = more premium collected = higher total yield. Lower min_dte to 7 days, target weekly Deribit expiries. More data also improves optimiser quality.',
         action='Set min_dte=7, max_dte=21, run evolution again'),
    dict(rank=2, effort=8, impact=9, color='#f59e0b',
         title='Add covered calls after assignment',
         detail='The full wheel includes selling calls when assigned BTC. Currently the bot only runs the put side. Adding covered calls after assignment reduces the cost basis significantly and adds another premium stream.',
         action='Build covered_call module: when holding BTC, auto-sell OTM call'),
    dict(rank=3, effort=5, impact=7, color='#3b82f6',
         title='Dynamic delta based on IV rank',
         detail='Sell more aggressive delta (closer to ATM) when IV rank is very high (option premium is richest). Pull back to more conservative delta when IV is low. Current config uses same delta regardless of market environment.',
         action='Map iv_rank → delta_target in a lookup table or linear interpolation'),
    dict(rank=4, effort=3, impact=5, color='#22c55e',
         title='Verify regime filter is active',
         detail=f'config.yaml has use_regime_filter and regime_ma_days={strat.get("regime_ma_days", "?")}. Confirm this is actually being applied in the bot logic. In a sustained downtrend, reduce position size or skip new entries.',
         action='Audit bot.py: confirm regime check gates trade entry'),
    dict(rank=5, effort=6, impact=5, color='#3b82f6',
         title='Strike laddering (multiple smaller legs)',
         detail='Instead of 1 put at 1 strike, sell 2-3 puts at different strikes (e.g. 8%, 10%, 12% OTM). Smooths the entry, avoids cliff-risk where a single level gets breached. Requires more BTC equity to fund.',
         action='Add multi_leg config: list of [offset, fraction] pairs'),
    dict(rank=6, effort=7, impact=6, color='#f59e0b',
         title='Roll losing positions before expiry',
         detail='When a put is breached (BTC falls below strike), consider rolling it: buy back the short put and re-sell at a lower strike further out in time. This collects more premium and buys time for BTC to recover.',
         action='Add roll_threshold: if pnl < -X% of premium, roll out and down'),
]

for imp in improvements:
    print(f'#{imp["rank"]} — {imp["title"]}')
    print(f'   What   : {imp["detail"]}')
    print(f'   Action : {imp["action"]}')
    print()

In [ ]:
# ── Priority matrix ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 8))

for imp in improvements:
    ax.scatter(imp['effort'], imp['impact'],
               s=350, color=imp['color'], alpha=0.9,
               edgecolors='white', linewidth=1.5, zorder=5)
    ax.annotate(f'#{imp["rank"]} {imp["title"]}',
                (imp['effort'], imp['impact']),
                textcoords='offset points', xytext=(10, 4),
                fontsize=9, color='white',
                bbox=dict(boxstyle='round,pad=0.2', fc=CARD, ec='#334155', alpha=0.85))

ax.axvline(5, color='#334155', linewidth=1, linestyle='--')
ax.axhline(6, color='#334155', linewidth=1, linestyle='--')
ax.text(1.2, 9.7, '🚀 Quick Wins',    color=ACCENT,  fontsize=11, fontstyle='italic')
ax.text(6.5, 9.7, '🏗  Big Projects', color=ACCENT2, fontsize=11, fontstyle='italic')
ax.text(1.2, 1.2, '😴 Low priority',  color='#64748b', fontsize=11, fontstyle='italic')
ax.text(6.5, 1.2, '❌ Avoid',          color=DANGER,  fontsize=11, fontstyle='italic')

ax.set_xlabel('Implementation Effort  (1 = easy, 10 = hard)', fontsize=11)
ax.set_ylabel('Expected Impact  (1 = low, 10 = high)',          fontsize=11)
ax.set_title('Strategy Improvement Priority Matrix', color='white', fontsize=13)
ax.set_xlim(0, 11)
ax.set_ylim(0, 11)
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig(BASE / 'data' / 'chart_improvement_matrix.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

---
## Summary

You now have a complete picture of the BTC Wheel Bot from first principles to production results.

| Area | Key finding |
|------|-------------|
| **Strategy** | Wheel (systematic short puts) harvesting BTC's elevated IV premium |
| **Edge** | BTC implied vol is persistently overpriced vs realised vol |
| **Backtest** | High win rate, positive cumulative P&L across 27 trades |
| **Walk-forward** | Robustness score 1.55 — strategy improved on unseen data ✅ |
| **Monte Carlo** | High probability of profit across 100 simulations |
| **Risk** | Max drawdown bounded; delta hedge reduces crash impact |
| **Top opportunity** | More trade frequency + full wheel (add covered calls) |

### Next steps for you as a developer of this strategy

1. **Understand** — re-run this notebook whenever you change config and see what shifts
2. **Experiment** — edit parameters and re-run sections 4 & 5 to see the visual impact before deploying
3. **Challenge** — look at the sweep sensitivity curves (Section 8) and ask 'why does this parameter peak here?'
4. **Improve** — tackle the quick wins from the priority matrix (Section 13), run the optimiser after each change
5. **Validate** — every improvement should pass walk-forward validation before going live

---
*Notebook generated from live data in `~/Documents/btc-wheel-bot/data/`*

*All charts saved as PNG files in `~/Documents/btc-wheel-bot/data/` for reference.*